# Grain Strategy Research Process - Concise Notebook

This notebook is a concise research-walkthrough, not a full rerun notebook.

Each code cell builds a clean table that explains:

- why the test was needed;
- what formula or model was tested;
- what result came out;
- what decision it justified.

The heavier experiment scripts and logs remain in the repo. This notebook is for understanding and presentation.

In [2]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)


def show(df):
    return df.style.format(precision=3, na_rep="-")

## 1. Universal Testing Ladder

Before creating product-specific strategies, I used the same basic ladder for each product. This prevents the research from jumping directly to a hand-picked final model.

In [3]:
testing_ladder = pd.DataFrame([
    [1, "Average all signals", "Lowest-overfit baseline", "If this works, no complex selection is needed."],
    [2, "Equal family weight", "Check signal-count dilution", "Each economic family gets equal influence."],
    [3, "IC family selection", "Check if predictive families can be selected", "Useful diagnostic, but validation IC can overfit."],
    [4, "Regime best family", "Check if edge changes by regime", "Tests trend/MR or vol-state dependence."],
    [5, "Regime avg families", "Reduce regime-selection risk", "Keeps regime idea but avoids one-family winner."],
    [6, "OLS/Kalman", "Benchmark dynamic fitted weights", "Reject if coefficient risk is not rewarded."],
    [7, "Ridge", "Benchmark regularized fitted weights", "Reject if OOS/full-period profile is unstable."],
    [8, "Performance analyst", "Find the real failure mode", "This determines the next product-specific test."],
], columns=["Step", "Test", "Why test it?", "What does it decide?"])
show(testing_ladder)

Matplotlib is building the font cache; this may take a moment.


,Step,Test,Why test it?,What does it decide?
0,1,Average all signals,Lowest-overfit baseline,"If this works, no complex selection is needed."
1,2,Equal family weight,Check signal-count dilution,Each economic family gets equal influence.
2,3,IC family selection,Check if predictive families can be selected,"Useful diagnostic, but validation IC can overfit."
3,4,Regime best family,Check if edge changes by regime,Tests trend/MR or vol-state dependence.
4,5,Regime avg families,Reduce regime-selection risk,Keeps regime idea but avoids one-family winner.
5,6,OLS/Kalman,Benchmark dynamic fitted weights,Reject if coefficient risk is not rewarded.
6,7,Ridge,Benchmark regularized fitted weights,Reject if OOS/full-period profile is unstable.
7,8,Performance analyst,Find the real failure mode,This determines the next product-specific test.


## 2. Generic Audit Results

This table is the first research checkpoint. It shows why generic methods were not enough.

In [4]:
generic_audit = pd.DataFrame([
    ["Soybeans", "Avg all signals", "mean(all provided signals)", 0.382, -342, "Low-overfit but diluted"],
    ["Soybeans", "Equal family weight", "mean(price, curve, COT, public physical, Cargill physical)", 0.233, -352, "Family balancing did not help"],
    ["Soybeans", "IC family selection", "select best train/validation IC family", 0.721, -684, "Higher Sharpe but DD too large"],
    ["Soybeans", "Regime best family", "best family by trend/MR regime", 1.005, -223, "Regime logic helped"],
    ["Soybeans", "OLS/Kalman", "dynamic linear coefficients", 0.357, -348, "Not better than economic blends"],
    ["Soybeans", "Ridge", "expanding Ridge, fixed alpha", 0.515, -698, "Positive OOS but unstable DD"],

    ["Corn", "Avg all signals", "mean(all provided signals)", -0.223, -437, "Generic corn signal failed"],
    ["Corn", "Equal family weight", "mean(price, curve, COT, public physical, Cargill physical)", -0.400, -534, "Family balancing failed"],
    ["Corn", "IC family selection", "select best train/validation IC family", -0.133, -618, "Validation IC did not transfer"],
    ["Corn", "Regime best family", "best family by trend/MR regime", -0.558, -461, "Trend/MR was wrong lens"],
    ["Corn", "OLS/Kalman", "dynamic linear coefficients", -0.390, -844, "Fitted model worsened risk"],
    ["Corn", "Ridge", "expanding Ridge, fixed alpha", -0.110, -794, "Still weak"],

    ["Wheat SRW", "Avg all signals", "mean(all SRW provided signals)", -0.471, -419, "Directional SRW failed"],
    ["Wheat HRW", "Avg all signals", "mean(all HRW provided signals)", -0.477, -359, "Directional HRW failed"],
    ["Wheat SRW", "Regime avg families", "avg families by trend/MR regime", 0.184, -200, "Slightly better but still weak"],
    ["Wheat HRW", "Regime best family", "best family by trend/MR regime", 0.030, -197, "Too weak as standalone"],
    ["Wheat SRW", "OLS/Kalman", "dynamic linear coefficients", 0.064, -441, "Not robust"],
    ["Wheat HRW", "OLS/Kalman", "dynamic linear coefficients", -0.554, -719, "Reject"],
    ["Wheat SRW", "Ridge", "expanding Ridge, fixed alpha", -0.292, -842, "Reject"],
    ["Wheat HRW", "Ridge", "expanding Ridge, fixed alpha", -0.032, -548, "Reject"],
], columns=["Product", "Test", "Formula", "OOS Sharpe", "OOS DD", "Decision"])
show(generic_audit)

,Product,Test,Formula,OOS Sharpe,OOS DD,Decision
0,Soybeans,Avg all signals,mean(all provided signals),0.382,-342,Low-overfit but diluted
1,Soybeans,Equal family weight,"mean(price, curve, COT, public physical, Cargill physical)",0.233,-352,Family balancing did not help
2,Soybeans,IC family selection,select best train/validation IC family,0.721,-684,Higher Sharpe but DD too large
3,Soybeans,Regime best family,best family by trend/MR regime,1.005,-223,Regime logic helped
4,Soybeans,OLS/Kalman,dynamic linear coefficients,0.357,-348,Not better than economic blends
5,Soybeans,Ridge,"expanding Ridge, fixed alpha",0.515,-698,Positive OOS but unstable DD
6,Corn,Avg all signals,mean(all provided signals),-0.223,-437,Generic corn signal failed
7,Corn,Equal family weight,"mean(price, curve, COT, public physical, Cargill physical)",-0.400,-534,Family balancing failed
8,Corn,IC family selection,select best train/validation IC family,-0.133,-618,Validation IC did not transfer
9,Corn,Regime best family,best family by trend/MR regime,-0.558,-461,Trend/MR was wrong lens


## 3. Soybean Thinking Process

The soybean story is: Cargill crush worked, external weather/crush/FX improved the economics, and the remaining issue was low-volatility drawdown control.

In [5]:
soybean_process = pd.DataFrame([
    [1, "Confirm provided-data edge", "given_crush_pressure = avg(crush_surprise, crush_utilization)", 1.604, 207, -95, "Keep Cargill crush as core evidence"],
    [2, "Check broader provided blend", "trend + inventory + cgl_inv + cgl_crush", 1.393, 227, -88, "Useful baseline but not best"],
    [3, "Benchmark fitted model", "OLS/Kalman dynamic coefficients", 0.357, 369, -348, "Reject as final; coefficient risk not rewarded"],
    [4, "Test direct weather value", "0.5 * Cargill crush + 0.5 * weather HDD/CDD", 2.639, 154, -34, "Best Sharpe/DD evidence"],
    [5, "Test broader economic blend", "40% physical + 20% FX + 20% external crush + 20% weather", 2.168, 222, -55, "Best fund-style base"],
    [6, "Diagnose weak periods", "period-performance analyst", None, -38, -71, "Low-price abundant supply was weak"],
    [7, "Add no-fit risk control", "if low_vol: position *= 0.5", 2.474, 162, -35, "Recommended soybean version"],
], columns=["Step", "Rationale", "Test / formula", "OOS Sharpe", "OOS PnL", "OOS DD", "Next decision"])
show(soybean_process)

,Step,Rationale,Test / formula,OOS Sharpe,OOS PnL,OOS DD,Next decision
0,1,Confirm provided-data edge,"given_crush_pressure = avg(crush_surprise, crush_utilization)",1.604,207,-95,Keep Cargill crush as core evidence
1,2,Check broader provided blend,trend + inventory + cgl_inv + cgl_crush,1.393,227,-88,Useful baseline but not best
2,3,Benchmark fitted model,OLS/Kalman dynamic coefficients,0.357,369,-348,Reject as final; coefficient risk not rewarded
3,4,Test direct weather value,0.5 * Cargill crush + 0.5 * weather HDD/CDD,2.639,154,-34,Best Sharpe/DD evidence
4,5,Test broader economic blend,40% physical + 20% FX + 20% external crush + 20% weather,2.168,222,-55,Best fund-style base
5,6,Diagnose weak periods,period-performance analyst,-,-38,-71,Low-price abundant supply was weak
6,7,Add no-fit risk control,if low_vol: position *= 0.5,2.474,162,-35,Recommended soybean version


## 4. Corn Thinking Process

The corn story is: generic signals failed, ethanol was relevant but unstable alone, and the main improvement came from guarding against abundant-supply drawdown.

In [6]:
corn_process = pd.DataFrame([
    [1, "Check generic provided data", "avg/equal-family provided signals", -0.223, -92, -437, "Reject generic corn signal"],
    [2, "Benchmark fitted model", "OLS/Kalman dynamic coefficients", -0.390, -349, -844, "Reject fitted model"],
    [3, "Test corn-specific demand", "EIA ethanol family", 0.653, 435, -333, "Relevant but unstable standalone"],
    [4, "Blend ethanol conservatively", "80% given + 10% ethanol + 10% FX", 0.353, 203, -426, "Requirement-compliant but weak"],
    [5, "Try better regime lens", "vol-regime IC strategy", 0.759, 224, -243, "Best early corn base"],
    [6, "Diagnose weak periods", "period-performance analyst", None, -124, -416, "Low-price abundant supply was main DD problem"],
    [7, "Add no-fit risk guard", "if price < 252d MA or mom_60 < 0: flat", 1.834, 266, -141, "Best corn version, small sleeve"],
], columns=["Step", "Rationale", "Test / formula", "OOS Sharpe", "OOS PnL", "OOS DD", "Next decision"])
show(corn_process)

,Step,Rationale,Test / formula,OOS Sharpe,OOS PnL,OOS DD,Next decision
0,1,Check generic provided data,avg/equal-family provided signals,-0.223,-92,-437,Reject generic corn signal
1,2,Benchmark fitted model,OLS/Kalman dynamic coefficients,-0.390,-349,-844,Reject fitted model
2,3,Test corn-specific demand,EIA ethanol family,0.653,435,-333,Relevant but unstable standalone
3,4,Blend ethanol conservatively,80% given + 10% ethanol + 10% FX,0.353,203,-426,Requirement-compliant but weak
4,5,Try better regime lens,vol-regime IC strategy,0.759,224,-243,Best early corn base
5,6,Diagnose weak periods,period-performance analyst,-,-124,-416,Low-price abundant supply was main DD problem
6,7,Add no-fit risk guard,if price < 252d MA or mom_60 < 0: flat,1.834,266,-141,"Best corn version, small sleeve"


## 5. Wheat SRW / HRW Thinking Process

The wheat story is: standalone directional wheat failed, so the correct structure was SRW/HRW relative value.

In [7]:
wheat_process = pd.DataFrame([
    [1, "Check standalone wheat", "best SRW and HRW directional sleeves", -0.044, None, -383, "Reject standalone directional wheat"],
    [2, "Check family/regime directional wheat", "best directional regime/family rows", 0.184, 88, -200, "Still too weak"],
    [3, "Benchmark fitted model", "OLS/Kalman dynamic coefficients", 0.064, 55, -441, "Reject fitted directional model"],
    [4, "Change economic structure", "90% SRW/HRW price MR + 10% Cargill physical", 1.129, 26.7, -19.9, "Recommended clean wheat base"],
    [5, "Diagnose trend weakness", "period-performance analyst", None, -14.4, -17.7, "COVID recovery hurt base MR"],
    [6, "Add small trend awareness", "80% MR/Cargill + 20% pair trend", 1.291, 36.4, -29.4, "Better in trend-like periods"],
    [7, "Test conservative risk overlay", "flat when pair trend conflicts with MR", 2.145, 26.7, -17.0, "Best DD, but more timing-like"],
], columns=["Step", "Rationale", "Test / formula", "OOS Sharpe", "OOS PnL", "OOS DD", "Next decision"])
show(wheat_process)

,Step,Rationale,Test / formula,OOS Sharpe,OOS PnL,OOS DD,Next decision
0,1,Check standalone wheat,best SRW and HRW directional sleeves,-0.044,-,-383.000,Reject standalone directional wheat
1,2,Check family/regime directional wheat,best directional regime/family rows,0.184,88.000,-200.000,Still too weak
2,3,Benchmark fitted model,OLS/Kalman dynamic coefficients,0.064,55.000,-441.000,Reject fitted directional model
3,4,Change economic structure,90% SRW/HRW price MR + 10% Cargill physical,1.129,26.700,-19.900,Recommended clean wheat base
4,5,Diagnose trend weakness,period-performance analyst,-,-14.400,-17.700,COVID recovery hurt base MR
5,6,Add small trend awareness,80% MR/Cargill + 20% pair trend,1.291,36.400,-29.400,Better in trend-like periods
6,7,Test conservative risk overlay,flat when pair trend conflicts with MR,2.145,26.700,-17.000,"Best DD, but more timing-like"


## 6. What To Present vs Appendix

This is the pruning table. It keeps the main story concise.

In [8]:
pruning = pd.DataFrame([
    ["Soybeans", "Present", "Cargill crush baseline; crush + weather; drawdown-priority blend; low-vol half exposure"],
    ["Soybeans", "Appendix", "Broad averages, OLS/Kalman/Ridge, broad external equal-weight, low-vol MR replacements"],
    ["Corn", "Present", "Generic methods failed; ethanol relevance; vol-regime base; abundant-supply guard"],
    ["Corn", "Appendix", "Flat IC winner, trend/MR regime failures, OLS/Kalman/Ridge"],
    ["Wheat", "Present", "Standalone wheat failed; SRW/HRW pair MR + Cargill; trend-aware pair variant"],
    ["Wheat", "Appendix", "Directional IC/regime wheat, fitted directional models, hard trend switches"],
], columns=["Product", "Use", "Items"])
show(pruning)

,Product,Use,Items
0,Soybeans,Present,Cargill crush baseline; crush + weather; drawdown-priority blend; low-vol half exposure
1,Soybeans,Appendix,"Broad averages, OLS/Kalman/Ridge, broad external equal-weight, low-vol MR replacements"
2,Corn,Present,Generic methods failed; ethanol relevance; vol-regime base; abundant-supply guard
3,Corn,Appendix,"Flat IC winner, trend/MR regime failures, OLS/Kalman/Ridge"
4,Wheat,Present,Standalone wheat failed; SRW/HRW pair MR + Cargill; trend-aware pair variant
5,Wheat,Appendix,"Directional IC/regime wheat, fitted directional models, hard trend switches"


## 7. Final Research Conclusion

The final conclusion is not that the most complex model wins. The research shows the opposite:

- Soybeans need direct physical/crush/weather/export economics plus a simple low-vol risk control.
- Corn needs corn-specific demand data and an abundant-supply guard, but should be sized smaller.
- Wheat should be traded as SRW/HRW relative value rather than standalone outright direction.

The strongest fund-style ideas are the ones with clear economic rationale and minimal fitted-weight overfit.